# Прогноз голов и угловых до начала матча

Запустите весь ноутбук из каталога проекта с `src/`, `data/core_matches.parquet`,
`data/open_matches.parquet` и таблицами `data/spring/`. Требуется установленное
окружение из requirements.txt, Python 3.12, CPU. Внешние данные не используются.

Конфигурация выбрана на development и фиксирована. Четыре модели обучаются до
01.03.2026; при прогнозе меняется только доступная история. Политика доступности
`kickoff + 24h < время прогноза` — допущение из-за отсутствия времён публикации.
Последняя ячейка создаёт `predictions_spring.parquet`.

In [ ]:
import os
import sys
from pathlib import Path

for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
             "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS", "LOKY_MAX_CPU_COUNT"):
    os.environ[name] = "6"
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Use the tested Python 3.12 environment")
ROOT = Path.cwd().resolve()
if not (ROOT / "src/quant_football").is_dir():
    raise FileNotFoundError("Run the notebook from the project directory containing src and data")
sys.path.insert(0, str(ROOT / "src"))
import json
import time
import pandas as pd
import quant_football
import resource

def peak_rss_gib():
    rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return rss / 2**30 if sys.platform == "darwin" else rss * 1024 / 2**30

from quant_football.config import TRAIN_CUTOFF, AvailabilityPolicy
from quant_football.forecasting import write_submission
from quant_football.io import ArtifactStore
from quant_football.stage6_model import FinalConfig, train_predict_final
if not Path(quant_football.__file__).resolve().is_relative_to(ROOT / "src"):
    raise RuntimeError("Imported package is outside the submission directory")
started = time.perf_counter()

In [ ]:
CONFIG = FinalConfig.from_dict({'catboost': {'depth': 6, 'l2_leaf_reg': 30.0, 'iterations': 300, 'learning_rate': 0.04, 'seed': 42, 'cohort': 'all_open', 'open_stats': True}, 'glm': {'kind': 'history_glm', 'alpha': 0.01, 'max_iter': 3000, 'tol': 1e-05}, 'glm_weight': 0.0})
POLICY = AvailabilityPolicy(lag_hours=24.0)
print(CONFIG.to_dict())

In [ ]:
paths = {"training_core": ROOT / "data/core_matches.parquet",
         "training_history": ROOT / "data/open_matches.parquet",
         "forecast_schedule": ROOT / "data/spring/core_matches.parquet",
         "forecast_history": ROOT / "data/spring/open_matches.parquet"}
missing = [str(p.relative_to(ROOT)) for p in paths.values() if not p.is_file()]
if missing:
    raise FileNotFoundError(f"Missing input files: {missing}")
frames = {name: pd.read_parquet(path) for name, path in paths.items()}
print({name: len(frame) for name, frame in frames.items()})

In [ ]:
prediction, audit = train_predict_final(**frames, cutoff=TRAIN_CUTOFF,
                                        config=CONFIG, policy=POLICY)
expected_fits = 8 if CONFIG.glm_weight else 4
if audit["model_fit_calls"] != expected_fits or audit["test_period_fit_calls"] != 0:
    raise RuntimeError("Unexpected fitting during forecast")
if audit["fingerprint_before"] != audit["fingerprint_after"]:
    raise RuntimeError("Model changed during prediction")
print({"rows": len(prediction), "model_fit_calls": audit["model_fit_calls"],
       "test_period_fit_calls": audit["test_period_fit_calls"]})

In [ ]:
expected_index = pd.Index(frames["forecast_schedule"].match_id, name="match_id")
submission = write_submission(ROOT / "predictions_spring.parquet", prediction, expected_index)
audit["submission"] = submission
audit["elapsed_seconds"] = time.perf_counter() - started
audit["peak_rss_gib"] = peak_rss_gib()
ArtifactStore(ROOT, ROOT / "data").json("run_audit.json", audit)
print(submission)
print({k: audit[k] for k in ("elapsed_seconds", "peak_rss_gib")})